In [1]:
import requests
import pandas as pd
import datetime as dt

API_KEY = "d2vhh09r01qm5lo8bhb0d2vhh09r01qm5lo8bhbg"
BASE_URL = "https://finnhub.io/api/v1/company-news"

def get_company_news(symbol, start_date, end_date):
    url = f"{BASE_URL}?symbol={symbol}&from={start_date}&to={end_date}&token={API_KEY}"
    response = requests.get(url).json()
    return pd.DataFrame(response)

# Example: Tesla news from last week
start = (dt.datetime.now() - dt.timedelta(days=7)).strftime("%Y-%m-%d")
end = dt.datetime.now().strftime("%Y-%m-%d")

df_news = get_company_news("TSLA", start, end)
print(df_news.head())


  category    datetime                                           headline  \
0  company  1761877866  Amazon's New AI Chips Could Unlock Billions In...   
1  company  1761876070  If You Invested $10K In LTC Properties Stock 1...   
2  company  1761874279  Trump Hints At Potential 'Very Large Scale' Al...   
3  company  1761870675  Jim Cramer Says 'I Want President Trump To Let...   
4  company  1761867087  Trump Concludes 'Amazing' Meeting With Xi Jinp...   

          id                                              image related  \
0  137274883  https://s.yimg.com/rz/stage/p/yahoo_finance_en...    TSLA   
1  137274941  https://s.yimg.com/rz/stage/p/yahoo_finance_en...    TSLA   
2  137274942  https://s.yimg.com/rz/stage/p/yahoo_finance_en...    TSLA   
3  137274943  https://s.yimg.com/rz/stage/p/yahoo_finance_en...    TSLA   
4  137274944  https://s.yimg.com/rz/stage/p/yahoo_finance_en...    TSLA   

  source                                            summary  \
0  Yahoo  Amazon.com In

In [2]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    scores = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return "Positive", compound
    elif compound <= -0.05:
        return "Negative", compound
    else:
        return "Neutral", compound

df_news['VADER_Sentiment'], df_news['VADER_Score'] = zip(*df_news['headline'].map(vader_sentiment))


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\shahd\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

labels = ["Negative", "Neutral", "Positive"]

def finbert_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    sentiment = labels[probs.argmax()]
    confidence = probs.max().item()
    return sentiment, confidence

df_news['FinBERT_Sentiment'], df_news['FinBERT_Confidence'] = zip(*df_news['headline'].map(finbert_sentiment))


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [4]:
print(df_news[['headline', 'FinBERT_Sentiment', 'FinBERT_Confidence']].head(10))
print(df_news[['headline', 'VADER_Sentiment', 'VADER_Score']].head(10))

                                            headline FinBERT_Sentiment  \
0  Amazon's New AI Chips Could Unlock Billions In...           Neutral   
1  If You Invested $10K In LTC Properties Stock 1...          Negative   
2  Trump Hints At Potential 'Very Large Scale' Al...          Negative   
3  Jim Cramer Says 'I Want President Trump To Let...          Negative   
4  Trump Concludes 'Amazing' Meeting With Xi Jinp...          Negative   
5  Ripple-Backed Evernorth Eyes $1B IPO To Become...          Negative   
6  Jensen Huang Says Nvidia Has Visibility Into '...           Neutral   
7  Trump's Pardon Of Changpeng Zhao Reportedly Fo...          Negative   
8  Was It Wise For Doge To Invest In Sports Clubs...          Negative   
9  The Dow Lost Steam. It's Hovering Near Breakeven.          Positive   

   FinBERT_Confidence  
0            0.548176  
1            0.999990  
2            0.803848  
3            0.999960  
4            0.999939  
5            0.886532  
6            0.94

In [5]:
# Core imports
from typing import List, Dict, Any, Optional
import requests
import pandas as pd
import numpy as np
import datetime as dt
import math

# Transformers + Torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

# NLTK (VADER sentiment)
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')

# spaCy for Named Entity Recognition (NER)
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except Exception:
    nlp = None  # if not available, run: !python -m spacy download en_core_web_sm



[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\shahd\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


In [19]:
# API configuration
FINNHUB_API_KEY = "d2vhh09r01qm5lo8bhb0d2vhh09r01qm5lo8bhbg"
FINNHUB_COMPANY_NEWS_URL = "https://finnhub.io/api/v1/company-news"

# Transformer model names
TRANSFORMER_MODEL_1 = "yiyanghkust/finbert-tone"   # Finance-specific
TRANSFORMER_MODEL_2 = "microsoft/deberta-base" # Strong general model

# Device (GPU if available)
DEVICE = 0 if torch.cuda.is_available() else -1



In [20]:
def timestamp_to_date(ts: int) -> str:
    return dt.datetime.utcfromtimestamp(ts).strftime("%Y-%m-%d")

def recency_weight(days_old: float, half_life_days: float = 7.0) -> float:
    """Exponential decay: more recent news has more weight"""
    return math.exp(-math.log(2) * (days_old / half_life_days))



In [21]:
class NewsSentimentEngine:
    def __init__(self, finnhub_key: Optional[str] = None,
                 model1: str = TRANSFORMER_MODEL_1,
                 model2: str = TRANSFORMER_MODEL_2,
                 device: int = DEVICE):

        self.finnhub_key = finnhub_key or FINNHUB_API_KEY
        self.vader = SentimentIntensityAnalyzer()

        # ----------------------------
        # Transformer 1 (FinBERT)
        # ----------------------------
        print("Loading FinBERT model...")
        self.tok1 = AutoTokenizer.from_pretrained(model1)
        self.model1 = AutoModelForSequenceClassification.from_pretrained(model1)
        self.pipe1 = pipeline("sentiment-analysis", model=self.model1,
                              tokenizer=self.tok1, device=device)

        # ----------------------------
        # Transformer 2 (DeBERTa)
        # ----------------------------
        print("Loading DeBERTa model...")
        try:
            # Try fast tokenizer
            self.tok2 = AutoTokenizer.from_pretrained(model2)
        except Exception as e:
            print(f"[Warning] Fast tokenizer failed: {e}")
            print("Retrying with use_fast=False ...")
            # Fallback to slow tokenizer
            self.tok2 = AutoTokenizer.from_pretrained(model2, use_fast=False)

        self.model2 = AutoModelForSequenceClassification.from_pretrained(model2)
        self.pipe2 = pipeline("sentiment-analysis", model=self.model2,
                              tokenizer=self.tok2, device=device)

        # ----------------------------
        # spaCy NER (optional)
        # ----------------------------
        global nlp
        if nlp is None:
            try:
                import spacy
                nlp = spacy.load("en_core_web_sm")
            except Exception as e:
                print(f"[Warning] spaCy model not loaded: {e}")
                nlp = None

    # ----------------------------
    # Data fetching
    # ----------------------------
    def fetch_company_news(self, symbol: str, start: str, end: str, limit: int = 50) -> pd.DataFrame:
        if not self.finnhub_key:
            raise ValueError("Finnhub API key not provided.")
        url = f"{FINNHUB_COMPANY_NEWS_URL}?symbol={symbol}&from={start}&to={end}&token={self.finnhub_key}"
        r = requests.get(url, timeout=10)
        r.raise_for_status()
        df = pd.DataFrame(r.json())
        if df.empty:
            return df
        if "datetime" in df.columns:
            df["datetime"] = pd.to_datetime(df["datetime"], unit="s")
        return df.head(limit)

    # ----------------------------
    # VADER
    # ----------------------------
    def vader_score(self, text: str) -> Dict[str, float]:
        s = self.vader.polarity_scores(text)
        return {"label": self._vader_label(s["compound"]),
                "compound": s["compound"], **s}

    def _vader_label(self, compound: float) -> str:
        if compound >= 0.05: return "POS"
        elif compound <= -0.05: return "NEG"
        else: return "NEU"

    # ----------------------------
    # Transformers
    # ----------------------------
    def transformer_scores(self, texts: List[str], pipe) -> List[Dict[str, Any]]:
        results = pipe(texts, truncation=True, batch_size=16)
        normalized = []
        for r in results:
            lbl = r["label"].upper()
            if "POS" in lbl: label = "POS"
            elif "NEG" in lbl: label = "NEG"
            elif "NEU" in lbl: label = "NEU"
            else: label = lbl
            normalized.append({"label": label, "score": float(r["score"])})
        return normalized

    # ----------------------------
    # NER
    # ----------------------------
    def extract_entities(self, text: str) -> List[str]:
        if nlp is None:
            return []
        doc = nlp(text)
        ents = [ent.text for ent in doc.ents if ent.label_ in ("ORG", "PRODUCT", "PERSON")]
        return list(dict.fromkeys(ents))

    # ----------------------------
    # Article analysis
    # ----------------------------
    def analyze_articles(self, df: pd.DataFrame, text_columns: List[str] = ["headline","summary"]) -> pd.DataFrame:
        if df.empty: return df

        def build_text(row):
            return " ".join(str(row[c]) for c in text_columns if c in row and pd.notnull(row[c]))[:4000]

        df = df.copy()
        df["primary_text"] = df.apply(build_text, axis=1)
        texts = df["primary_text"].tolist()

        # Sentiment predictions
        m1_out = self.transformer_scores(texts, self.pipe1)
        m2_out = self.transformer_scores(texts, self.pipe2)
        vader_out = [self.vader_score(t) for t in texts]

        # Combine outputs
        df["entities"] = df["primary_text"].apply(self.extract_entities)
        df["model1_label"] = [o["label"] for o in m1_out]
        df["model1_score"] = [o["score"] for o in m1_out]
        df["model2_label"] = [o["label"] for o in m2_out]
        df["model2_score"] = [o["score"] for o in m2_out]
        df["vader_label"] = [o["label"] for o in vader_out]
        df["vader_compound"] = [o["compound"] for o in vader_out]

        # Simple ensemble
        df["ensemble_score"] = df.apply(
            lambda r: 0.45*self._label_to_numeric(r["model1_label"], r["model1_score"]) +
                      0.35*self._label_to_numeric(r["model2_label"], r["model2_score"]) +
                      0.20*r["vader_compound"], axis=1)
        return df

    def _label_to_numeric(self, label: str, score: float) -> float:
        if label == "POS": return score
        elif label == "NEG": return -score
        else: return 0.0

    # ----------------------------
    # Aggregation
    # ----------------------------
    def aggregate_for_symbol(self, df_articles: pd.DataFrame, target_symbol: str, top_k: int = 10) -> Dict[str, Any]:
        if df_articles.empty:
            return {"features": {}, "top_articles": pd.DataFrame(), "raw": df_articles}

        cond = df_articles["primary_text"].str.contains(target_symbol, case=False, na=False)
        filtered = df_articles[cond].copy()
        if filtered.empty:
            filtered = df_articles.copy()

        now = pd.Timestamp.utcnow()
        if "datetime" in filtered.columns:
            filtered["days_old"] = (now - filtered["datetime"]).dt.total_seconds() / 86400
        else:
            filtered["days_old"] = 0.0
        filtered["recency_w"] = filtered["days_old"].apply(lambda d: recency_weight(d))

        filtered["weighted_score"] = filtered["ensemble_score"] * filtered["recency_w"]
        recency_mean = filtered["weighted_score"].sum() / (filtered["recency_w"].sum() or 1)

        pos_count = (filtered["ensemble_score"] > 0.05).sum()
        neg_count = (filtered["ensemble_score"] < -0.05).sum()
        neu_count = len(filtered) - pos_count - neg_count

        features = {
            "recency_weighted_sentiment": float(recency_mean),
            "pos_count": int(pos_count),
            "neg_count": int(neg_count),
            "neu_count": int(neu_count),
            "num_articles": int(len(filtered))
        }

        topk = filtered.sort_values("ensemble_score", key=np.abs, ascending=False).head(top_k)
        return {"features": features, "top_articles": topk, "raw": filtered}

    # ----------------------------
    # Main wrapper
    # ----------------------------
    def analyze_symbol(self, symbol: str, days: int = 7, top_k: int = 5):
        end_date = dt.datetime.utcnow().date()
        start_date = end_date - dt.timedelta(days=days)

        df = self.fetch_company_news(symbol, str(start_date), str(end_date))
        if df.empty:
            return {"features": {}, "top_articles": pd.DataFrame(), "all_articles": pd.DataFrame()}

        df = self.analyze_articles(df)

        features = {
            "mean_ensemble": float(df["ensemble_score"].mean()),
            "median_ensemble": float(df["ensemble_score"].median()),
            "positive_ratio": float((df["ensemble_score"] > 0.05).mean()),
            "negative_ratio": float((df["ensemble_score"] < -0.05).mean()),
            "article_count": int(len(df))
        }

        df_sorted = df.sort_values("ensemble_score", key=lambda x: x.abs(), ascending=False)
        top_articles = df_sorted.head(top_k)
        all_articles = df.sort_values("datetime", ascending=False)

        return {
            "features": features,
            "top_articles": top_articles[["datetime", "headline", "summary", "ensemble_score"]],
            "all_articles": all_articles[["datetime", "headline", "summary", "ensemble_score"]]
        }



# ==============================================================
# USAGE EXAMPLE
# ==============================================================
engine = NewsSentimentEngine(FINNHUB_API_KEY)
result = engine.analyze_symbol("TSLA", top_k=5)

print("Aggregated Features:", result["features"])

print("\nTop Articles (strongest sentiment):")
print(result["top_articles"][["headline", "ensemble_score"]].to_string(index=False))

print("\nAll Latest Articles:")
print(result["all_articles"][["headline", "ensemble_score"]].to_string(index=False))



Loading FinBERT model...


Device set to use cpu


Loading DeBERTa model...


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/474 [00:00<?, ?B/s]

...

[Warning] spaCy model not loaded: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/559M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Aggregated Features: {'mean_ensemble': 0.02875786675720215, 'median_ensemble': 0.04526, 'positive_ratio': 0.48, 'negative_ratio': 0.38, 'article_count': 50}

Top Articles (strongest sentiment):
                                                                                                        headline  ensemble_score
                               Energy Transition Update - Solar and Wind Propel Shift to Affordable Clean Energy        0.637571
                                                                          Is Apple the Best Magnificent 7 Stock?        0.621540
                                          Tesla stock pressured amid tech weakness and robotaxi rollout concerns       -0.601580
Trump's Pardon Of Changpeng Zhao Reportedly Followed Binance's High Level Task Force, $2 Billion Stablecoin Deal        0.598480
                          'Stay Nimble': Standard Chartered Says Bitcoin Is Set For 'Inevitable Dip' Below $100K       -0.577326

All Latest Articles:
          

In [22]:
import os

symbol = "TSLA"
output_dir = "news_sentiment_results"
os.makedirs(output_dir, exist_ok=True)

# 1️⃣ Save all articles (with sentiment scores)
all_articles_path = os.path.join(output_dir, f"{symbol}_all_articles.csv")
result["all_articles"].to_csv(all_articles_path, index=False, encoding="utf-8-sig")

# 2️⃣ Save top articles
top_articles_path = os.path.join(output_dir, f"{symbol}_top_articles.csv")
result["top_articles"].to_csv(top_articles_path, index=False, encoding="utf-8-sig")

# 3️⃣ Save aggregated features (as one-row CSV)
import pandas as pd
features_path = os.path.join(output_dir, f"{symbol}_features.csv")
pd.DataFrame([result["features"]]).to_csv(features_path, index=False, encoding="utf-8-sig")

print(f"\n✅ CSV files saved to folder: '{output_dir}'")
print(f" - {symbol}_all_articles.csv")
print(f" - {symbol}_top_articles.csv")
print(f" - {symbol}_features.csv")


✅ CSV files saved to folder: 'news_sentiment_results'
 - TSLA_all_articles.csv
 - TSLA_top_articles.csv
 - TSLA_features.csv


## NEW

In [23]:
import pandas as pd
import requests

query = "Apple Inc"
url = f"https://api.gdeltproject.org/api/v2/doc/doc?query={query}&mode=artlist&maxrecords=250&format=json"
res = requests.get(url).json()

articles = res.get("articles", [])
df = pd.DataFrame(articles)

df = df[["title", "url", "seendate", "sourcecountry", "language"]]
df["seendate"] = pd.to_datetime(df["seendate"])

print(df.head())


                                               title  \
0  蘋果起訴華裔前員工 竊取商業機密跳槽Oppo | 蘋果華裔員工 | OPPO公司 | 健康傳感技術   
1                              王文濤會見蘋果公司首席執行官庫克 - 首頁   
2  Apple ( AAPL ) Rebounded as Investor Concerns ...   
3  Apple ( AAPL ) Rebounded as Investor Concerns ...   
4                 中國工信部長會蘋果CEO ： 為蘋果等外企營造良好營商環境 | 電訊   

                                                 url  \
0  https://www.ntdtv.com/b5/2025/08/23/a104013968...   
1  https://www.wenweipo.com/a/202510/16/AP68f1029...   
2  https://finance.yahoo.com/news/apple-aapl-rebo...   
3  https://www.insidermonkey.com/blog/apple-aapl-...   
4  https://www.hkcna.hk/docDetail.jsp?id=10113629...   

                   seendate  sourcecountry language  
0 2025-08-24 00:45:00+00:00          China  Chinese  
1 2025-10-16 18:45:00+00:00          China  Chinese  
2 2025-10-10 20:45:00+00:00  United States  English  
3 2025-10-09 12:45:00+00:00  United States  English  
4 2025-10-15 10:15:00+00:00      Hong Kong  Chinese  


In [25]:
import pandas as pd
import requests
import time
import json
from datetime import datetime, timedelta

def get_gdelt_news(query, start_date, end_date):
    all_data = []
    date = start_date
    while date <= end_date:
        next_date = date + timedelta(days=7)  # fetch week by week
        url = (
            f"https://api.gdeltproject.org/api/v2/doc/doc?"
            f"query={query}&mode=artlist&maxrecords=250&format=json&"
            f"STARTDATETIME={date.strftime('%Y%m%d%H%M%S')}&"
            f"ENDDATETIME={next_date.strftime('%Y%m%d%H%M%S')}"
        )
        try:
            res = requests.get(url, timeout=20)
            if res.status_code == 200:
                try:
                    # Try to decode JSON safely
                    data = res.json().get("articles", [])
                    all_data.extend(data)
                except json.JSONDecodeError:
                    print(f"[Warning] Skipping invalid JSON between {date.date()} - {next_date.date()}")
                    continue
            else:
                print(f"[Warning] Request failed with {res.status_code} for {date.date()}")
        except Exception as e:
            print(f"[Error] {e}")
        time.sleep(1)  # avoid rate limits
        date = next_date
    return pd.DataFrame(all_data)


# Run safely
news_df = get_gdelt_news("Apple Inc", datetime(2024, 1, 1), datetime(2025, 1, 1))

if not news_df.empty:
    news_df.to_csv("apple_news_gdelt.csv", index=False)
    print(f"✅ Saved {len(news_df)} articles to apple_news_gdelt.csv")
else:
    print("⚠️ No data retrieved — try shorter date ranges or another query.")


[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 2024-03-11 - 2024-03-18
[Warning] Skipping invalid JSON between 

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import torch
import requests
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings

warnings.filterwarnings("ignore")

# ==========================================================
# CONFIG
# ==========================================================
DEVICE = 0 if torch.cuda.is_available() else -1
TRANSFORMER_MODEL_1 = "ProsusAI/finbert"
TRANSFORMER_MODEL_2 = "microsoft/deberta-v3-base"


# ==========================================================
# MAIN CLASS
# ==========================================================
class NewsSentimentEngine:
    def __init__(self,
                 model1: str = TRANSFORMER_MODEL_1,
                 model2: str = TRANSFORMER_MODEL_2,
                 device: int = DEVICE):

        self.vader = SentimentIntensityAnalyzer()
        print("✅ Initializing sentiment models...")

        # ----------------------------
        # FinBERT
        # ----------------------------
        print("Loading FinBERT...")
        self.tok1 = AutoTokenizer.from_pretrained(model1)
        self.model1 = AutoModelForSequenceClassification.from_pretrained(model1)
        self.pipe1 = pipeline("sentiment-analysis", model=self.model1,
                              tokenizer=self.tok1, device=device)

        # ----------------------------
        # DeBERTa
        # ----------------------------
        print("Loading DeBERTa...")
        self.tok2 = AutoTokenizer.from_pretrained(model2)
        self.model2 = AutoModelForSequenceClassification.from_pretrained(model2)
        self.pipe2 = pipeline("sentiment-analysis", model=self.model2,
                              tokenizer=self.tok2, device=device)

        print("✅ Models loaded successfully.\n")

    # ==========================================================
    # SENTIMENT PIPELINES
    # ==========================================================
    def vader_score(self, text: str) -> Dict[str, float]:
        s = self.vader.polarity_scores(text)
        return {"label": self._vader_label(s["compound"]),
                "compound": s["compound"], **s}

    def _vader_label(self, compound: float) -> str:
        if compound >= 0.05: return "POS"
        elif compound <= -0.05: return "NEG"
        else: return "NEU"

    def transformer_scores(self, texts: List[str], pipe) -> List[Dict[str, Any]]:
        results = pipe(texts, truncation=True, batch_size=16)
        normalized = []
        for r in results:
            lbl = r["label"].upper()
            if "POS" in lbl: label = "POS"
            elif "NEG" in lbl: label = "NEG"
            elif "NEU" in lbl: label = "NEU"
            else: label = lbl
            normalized.append({"label": label, "score": float(r["score"])})
        return normalized

    def _label_to_numeric(self, label: str, score: float) -> float:
        if label == "POS": return score
        elif label == "NEG": return -score
        else: return 0.0

    # ==========================================================
    # MARKET RELEVANCE LAYER
    # ==========================================================
    def assign_relevance(self, text: str, company: str, company_sector: str) -> int:
        text = text.lower()
        sector_keywords = {
            "tech": ["chip", "ai", "software", "data", "cloud", "semiconductor", "internet", "iphone", "device"],
            "auto": ["ev", "battery", "vehicle", "car", "autonomous", "tesla"],
            "macro": ["inflation", "interest", "fed", "market", "recession", "trade", "stocks", "economy"]
        }

        if company.lower() in text:
            return 3  # direct mention
        elif any(word in text for word in sector_keywords.get(company_sector, [])):
            return 2  # same sector
        elif any(word in text for word in sector_keywords["macro"]):
            return 1  # macro/global
        else:
            return 0  # irrelevant

    def compute_intensity_weight(self, sentiment_score: float) -> float:
        abs_score = abs(sentiment_score)
        if abs_score > 0.7:
            return 1.2
        elif abs_score > 0.3:
            return 1.0
        else:
            return 0.8

    def compute_final_score(self, sentiment: float, relevance: int) -> float:
        if relevance == 0:
            return 0.0
        intensity_w = self.compute_intensity_weight(sentiment)
        relevance_w = [0.0, 0.4, 0.7, 1.0][relevance]
        return sentiment * relevance_w * intensity_w

    # ==========================================================
    # MAIN ANALYSIS ON CSV DATA
    # ==========================================================
    def analyze_csv(self, csv_path: str, companies: Dict[str, str], output_path: str):
        df = pd.read_csv(csv_path)
        df = df.dropna(subset=["headline"])
        print(f"📰 Loaded {len(df)} headlines from {csv_path}")

        # Build text list for sentiment
        texts = df["headline"].astype(str).tolist()

        print("🔍 Running FinBERT sentiment...")
        m1_out = self.transformer_scores(texts, self.pipe1)
        print("🔍 Running DeBERTa sentiment...")
        m2_out = self.transformer_scores(texts, self.pipe2)
        print("🔍 Running VADER sentiment...")
        vader_out = [self.vader_score(t) for t in texts]

        df["model1_label"] = [o["label"] for o in m1_out]
        df["model1_score"] = [o["score"] for o in m1_out]
        df["model2_label"] = [o["label"] for o in m2_out]
        df["model2_score"] = [o["score"] for o in m2_out]
        df["vader_compound"] = [o["compound"] for o in vader_out]

        df["ensemble_score"] = df.apply(
            lambda r: 0.45*self._label_to_numeric(r["model1_label"], r["model1_score"]) +
                      0.35*self._label_to_numeric(r["model2_label"], r["model2_score"]) +
                      0.20*r["vader_compound"], axis=1)

        # Process for each company
        all_results = []
        for company, sector in companies.items():
            temp = df.copy()
            temp["relevance"] = temp["headline"].apply(lambda x: self.assign_relevance(str(x), company, sector))
            temp["final_weighted_score"] = temp.apply(
                lambda r: self.compute_final_score(r["ensemble_score"], r["relevance"]), axis=1
            )
            temp["company"] = company
            all_results.append(temp)

        final_df = pd.concat(all_results)
        final_df.to_csv(output_path, index=False)
        print(f"✅ Saved final results with sentiment and relevance weights → {output_path}")
        return final_df


# ==========================================================
# RUN EXAMPLE
# ==========================================================
if __name__ == "__main__":
    engine = NewsSentimentEngine()

    companies = {
        "AAPL": "tech",
        "MSFT": "tech",
        "TSLA": "auto"
    }

    final = engine.analyze_csv(
        csv_path=r"C:\Users\shahd\Desktop\major project\filtered_broad_news.csv",
        companies=companies,
        output_path=r"C:\Users\shahd\Desktop\major project\final_weighted_sentiment.csv"
    )

    print(final[["date", "headline", "company", "ensemble_score", "relevance", "final_weighted_score"]].head(10))


✅ Initializing sentiment models...
Loading FinBERT...


Device set to use cpu


Loading DeBERTa...


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

📰 Loaded 45902 headlines from C:\Users\shahd\Desktop\major project\filtered_broad_news.csv
🔍 Running FinBERT sentiment...


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


🔍 Running DeBERTa sentiment...


KeyboardInterrupt: 